## Exploración inicial de datos

### By:
Maria Camila Aristizábal Aguirre

### Date:
2026-08-16

### Description:
Exploración inicial: tipos de datos, unificación de valores nulos y persistencia en formato parquet.

## 📊 Importar librerias




In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa

DATA_DIR = Path("../../data")

## 💾 Load data

In [2]:
df = pd.read_csv(DATA_DIR / "01_raw" / "diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6.0,148.0,72.0,35.0,0.0,33.6,627.0,50.0,1.0
1,1.0,85.0,66.0,29.0,0.0,26.6,351.0,31.0,0.0
2,8.0,183.0,64.0,0.0,0.0,23.3,672.0,32.0,1.0
3,1.0,89.0,66.0,23.0,94.0,28.1,167.0,21.0,0.0
4,0.0,137.0,40.0,35.0,168.0,43.1,2288.0,33.0,1.0


## Inspección inicial

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 994 entries, 0 to 993
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               986 non-null    float64
 1   Glucose                   990 non-null    float64
 2   BloodPressure             989 non-null    float64
 3   SkinThickness             987 non-null    float64
 4   Insulin                   986 non-null    float64
 5   BMI                       992 non-null    float64
 6   DiabetesPedigreeFunction  989 non-null    float64
 7   Age                       991 non-null    float64
 8   Outcome                   975 non-null    float64
dtypes: float64(9)
memory usage: 70.0 KB


In [4]:
df.sample(10, random_state=42)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
919,10.0,129.0,76.0,28.0,122.0,35.9,0.28,39.0,0.0
525,3.0,87.0,60.0,18.0,0.0,21.8,444.00,21.0,0.0
567,6.0,92.0,62.0,32.0,126.0,32.0,85.00,46.0,0.0
656,2.0,101.0,58.0,35.0,90.0,21.8,155.00,22.0,0.0
926,4.0,96.0,56.0,17.0,49.0,20.8,0.34,26.0,0.0
429,1.0,95.0,82.0,25.0,180.0,35.0,233.00,43.0,1.0
869,6.0,134.0,80.0,37.0,370.0,46.2,238.00,46.0,1.0
711,5.0,126.0,78.0,27.0,22.0,29.6,439.00,40.0,0.0
174,2.0,75.0,64.0,24.0,55.0,29.7,0.37,33.0,0.0
604,4.0,183.0,0.0,0.0,0.0,28.4,212.00,36.0,1.0


## Unificación de valores nulos

En este dataset los valores faltantes están representados de dos formas distintas:
celdas vacías (ya son `NaN`) y ceros en columnas donde fisiológicamente
0 no es un valor válido (Glucose, BloodPressure, SkinThickness, Insulin, BMI).
Unificamos ambas representaciones a `NaN`.

Adicionalmente, `DiabetesPedigreeFunction` tiene un error de formato: a la mayoría
de los valores se les perdió el punto decimal (ej. 627 en vez de 0.627). Se corrige
dividiendo entre 1000 los valores mayores a 3.

In [7]:
COLUMNAS_SIN_CERO_VALIDO = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

df[COLUMNAS_SIN_CERO_VALIDO] = df[COLUMNAS_SIN_CERO_VALIDO].replace(0, np.nan)

df.isna().sum()

Pregnancies                   8
Glucose                      12
BloodPressure                52
SkinThickness               305
Insulin                     493
BMI                          16
DiabetesPedigreeFunction      5
Age                           3
Outcome                      19
dtype: int64

In [6]:
UMBRAL_PUNTO_DECIMAL_PERDIDO = 3

mascara_dpf_mal_formateado = df["DiabetesPedigreeFunction"] > UMBRAL_PUNTO_DECIMAL_PERDIDO
df.loc[mascara_dpf_mal_formateado, "DiabetesPedigreeFunction"] /= 1000

df["DiabetesPedigreeFunction"].describe()

count    989.000000
mean       0.473155
std        0.327575
min        0.078000
25%        0.248000
50%        0.371000
75%        0.619000
max        2.420000
Name: DiabetesPedigreeFunction, dtype: float64

## Clasificación y conversión de tipos de datos

- **Numéricas discretas** (conteos enteros): `Pregnancies`, `Age`
- **Numéricas continuas**: `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, `DiabetesPedigreeFunction`
- **Booleana** (variable objetivo): `Outcome`

Se usan tipos *nullable* de pandas (`Int8`, `boolean`) para conservar los `NaN`
existentes sin perder la semántica de entero/booleano.

In [8]:
COLUMNAS_ENTERAS = ["Pregnancies", "Age"]
COLUMNAS_CONTINUAS = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
]

df[COLUMNAS_ENTERAS] = df[COLUMNAS_ENTERAS].astype("Int8")
df[COLUMNAS_CONTINUAS] = df[COLUMNAS_CONTINUAS].astype("float64")
df["Outcome"] = df["Outcome"].astype("boolean")

df.dtypes

Pregnancies                    Int8
Glucose                     float64
BloodPressure               float64
SkinThickness               float64
Insulin                     float64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                            Int8
Outcome                     boolean
dtype: object

## Validación de esquema

In [9]:
schema = pa.Table.from_pandas(df).schema
schema

Pregnancies: int8
Glucose: double
BloodPressure: double
SkinThickness: double
Insulin: double
BMI: double
DiabetesPedigreeFunction: double
Age: int8
Outcome: bool
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1346

## Persistencia

In [10]:
ruta_salida = DATA_DIR / "02_intermediate" / "diabetes_type_fixed.parquet"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(ruta_salida, index=False, schema=schema)